# Sustained CPU and Akida measurements
Rows summarize prepared-input measurements per saved model. CPU package and Akida SDK sensor figures refer to separate electrical domains. API latency and the interval derived from the Akida inference counter cover different portions of execution.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"data/main_fit_metrics.csv").is_file())
OUT = Path(os.environ.get("ARTICLE_RESULTS_DIR", root/"generated")).resolve()
OUT.mkdir(parents=True, exist_ok=True)

def read(name):
    return pd.read_csv(root/"data"/(name+".csv"), float_precision="round_trip")

def save(name, frame):
    frame.to_csv(OUT/(name+".csv"), index=False)

def close(value, expected, tolerance):
    return abs(value-expected) <= tolerance

In [ ]:
timing = read("sustained_timing")
energy = read("sustained_energy")
resources = read("measurement_model_resources")
prediction = read("measurement_stage_metrics")
assert len(timing) == len(resources) == len(prediction) == 60
assert timing.groupby("config_order").size().eq(3).all()
assert len(energy) == 120 and energy.groupby("measurement_id").backend.nunique().eq(2).all()
assert resources.external_memory_bytes.eq(0).all() and resources.hardware_sequences.eq(1).all()
columns = ["cpu_single_api_ms", "akida_single_api_ms", "akida_single_sdk_clock_ms",
           "cpu_batch47_api_ms", "akida_batch47_api_ms", "akida_batch47_sdk_clock_ms",
           "single_outside_counter_fraction", "batch_outside_counter_fraction"]
summary = timing[columns].quantile([.25,.5,.75]).T
assert [round(timing[c].median(), 3) for c in columns[:6]] == [.269, .727, .017, .365, 17.254, .782]
assert round(timing.akida_single_sdk_clock_ms.median()*1000, 1) == 16.6
assert timing.cpu_batch47_api_ms.lt(timing.akida_batch47_sdk_clock_ms).sum() == 54
save("sustained_timing_summaries", summary.reset_index(names="measurement"))
save("sustained_power_summaries", energy.groupby("backend").median(numeric_only=True).reset_index())

x = energy.pivot(index="measurement_id", columns="backend", values="cpu_package_gross_mJ_input")
ratio = x.akida / x.cpu
akida = energy[energy.backend.eq("akida")]
assert close(ratio.median(), 36.0343, .0001)
assert close(akida.sensor_idle_allocation_fraction.median(), .9998, .00005)
assert close(akida.sensor_active_W.median(), .881292, .0000005)
assert close(akida.sensor_idle_W.median(), .881154, .0000005)
assert close(akida.cpu_package_active_W.median(), 15.958, .0005)
assert close(energy[energy.backend.eq("cpu")].cpu_package_active_W.median(), 24.392, .0005)
save("measurement_panel_predictive_means", prediction.mean(numeric_only=True).rename("mean").rename_axis("metric").reset_index())
print(f"Median paired host-package energy-per-input ratio: {ratio.median():.2f}")
display(summary)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(7.1, 5))
for ax, values, name in zip(axes, [columns[:3], columns[3:6]], ["One input", "47-input batch"]):
    labels = ["CPU API", "Akida API", "Akida counted interval"]
    ax.boxplot([timing[c] for c in values], vert=False, labels=labels)
    ax.set(xscale="log", xlabel="Time (ms)", title=name)
fig.tight_layout()
fig.savefig(OUT/"sustained_timing.png", dpi=300, bbox_inches="tight")
plt.close(fig)